
# 02 — Review-Based Question Answering

## Product-Specific Evidence Retrieval

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline for answering questions about products using real customer reviews.

The first experiment verifies that BM25 retrieves relevant reviews from the requested product only.

The retrieved reviews will be used as evidence for grounded answer generation, with explicit review ID citations.


In [1]:
from pathlib import Path

from digikala_assistant.retrieval.bm25 import BM25Retriever

cwd = Path.cwd().resolve()

project_root = next(
    path
    for path in [cwd, *cwd.parents]
    if (path / "pyproject.toml").exists()
)

sample_dir = project_root / "data" / "sample"

review_retriever = BM25Retriever(
    products_path=sample_dir / "products_dev.csv",
    reviews_path=sample_dir / "reviews_dev.csv",
)

product_id = 21342
question = "خریدارها درباره قدرت ماساژ این دستگاه چه نظری دارن؟"

review_hits = review_retriever.search_reviews(
    question,
    product_id=product_id,
    top_k=8,
)

assert review_hits
assert all(
    hit.item.product_id == product_id
    for hit in review_hits
)

print(f"Product ID: {product_id}")
print(f"Retrieved reviews: {len(review_hits)}")

for hit in review_hits:
    print(f"\n[review_id:{hit.item.review_id}]")
    print(f"Score: {hit.score:.4f}")
    print(hit.item.body[:350])


Product ID: 21342
Retrieved reviews: 8

[review_id:35575243]
Score: 5.4580
پیشنهاد میکنم حتما بخرید.
این دستگاه در فیزیوتراپی استفاده می شود.
حتما با ژل ماساژ انجام بدید. 
عالیهه
خیلی دوستش دارم

[review_id:2183681]
Score: 4.3210
من اینو برای همسرم گرفتم. خیلی از خریدم راضیم
چون هم راحت میشه ازش استفاده کرد هم اینکه خوب ماساژ میده. برعکس خیلی از ماساژور ها قدرت خوبی داره

[review_id:2093806]
Score: 4.3194
من این دستگاه رو هدیه گرفتم ،قدرت خوبی داره واقعا، اما کارایی مادون قرمزش رو نمیدونم دقیقا چی هست! اگه توضیح بیشتر میدادن بهتر بود چون یه سری میگن باید حرارت داشته باشه که این مدل اصلا حرارت نداره !

[review_id:33780671]
Score: 4.1646
من از این مارک قبلا خریده ام و عملکرد خوبی داشت. برای استفاده از هد مادون قرمز یا باصطلاح حرارتی باید دور دستگاه کم و بصورت نرکت خیلی کند ماساژ را انجام دهید. نسمه دستی یونیت به شما اجازه انواع حرکات را می دهد اما بخاطر نبود دمپر در ماساژهای طولانی دستتان بسیار حساس می شود. حد اکثر زمان ماساژ قبل ازگرم شدن دستگاه 20 دقیقه است..

[review_id:11118949]
Scor


## Grounded Answer Generation with RAG

This experiment uses product-specific review evidence to generate a natural-language answer through an LLM.

The pipeline validates citation integrity by ensuring that every cited review ID belongs to the retrieved evidence.

The retrieved reviews and generated answer are displayed together to support manual Grounding Evaluation. Citation validity alone does not guarantee factual correctness.


In [2]:
import os

from dotenv import load_dotenv
from openai import OpenAI

from digikala_assistant.review_qa.service import ReviewQAService

load_dotenv(project_root / ".env", override=True)

api_key = os.getenv("QBC13_API_KEY")

if not api_key:
    raise RuntimeError("QBC13_API_KEY is not configured.")

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv(
        "QBC13_API_BASE_URL",
        "https://193.93.169.12/v1",
    ),
    timeout=60.0,
    max_retries=2,
)

qa_service = ReviewQAService(
    backend=review_retriever,
    client=client,
)

result = qa_service.answer(
    question,
    product_id=product_id,
    top_k=8,
)

assert result.evidence
assert result.cited_review_ids

assert set(result.cited_review_ids).issubset({
    hit.item.review_id for hit in result.evidence
})

print("GENERATED ANSWER")
print("-" * 60)
print(result.answer)

print("\nCITED EVIDENCE")
print("-" * 60)

for hit in result.evidence:
    if hit.item.review_id in result.cited_review_ids:
        print(f"\n[review_id:{hit.item.review_id}]")
        print(hit.item.body[:700])

print("\nVALIDATION")
print(f"Retrieved reviews: {len(result.evidence)}")
print(f"Cited reviews: {len(result.cited_review_ids)}")
print(f"LLM tokens: {result.total_tokens}")

print("\nReview-Based QA validation passed.")


GENERATED ANSWER
------------------------------------------------------------
نظرات خریداران درباره قدرت ماساژ دستگاه "ماساژور برقی بیورر مدل MG70" متفاوت است:

1. برخی کاربران از قدرت ماساژ این دستگاه راضی هستند و آن را "عالی" و "با قدرت" توصیف کرده‌اند. به عنوان مثال، یکی از خریداران می‌گوید: "خیلی از خریدم راضیم چون هم راحت میشه ازش استفاده کرد هم اینکه خوب ماساژ میده" [review_id: 2183681] و دیگری اشاره می‌کند: "عالی ، با قدرت ، کارآمد ، خیلی راضیم ازش" [review_id: 42335026].

2. در مقابل، برخی دیگر از کاربران از قدرت ماساژ دستگاه ناراضی هستند. یکی از آن‌ها می‌گوید: "خیلی قدرت ضعیفی داره انگار اسباب بازیه" [review_id: 41439382]. همچنین، کاربری دیگر اشاره کرده که "قدرت خوبی داره واقعا" اما در عین حال به عدم وجود حرارت در دستگاه انتقاد کرده است [review_id: 2093806].

به طور کلی، نظرات متناقضی درباره قدرت ماساژ این دستگاه وجود دارد و برخی کاربران از آن راضی و برخی دیگر ناراضی هستند.

CITED EVIDENCE
------------------------------------------------------------

[review_id:2183681]
من این


## Multi-Query RAG Evaluation

This experiment evaluates Review-Based QA across multiple questions about the same product.

The evaluation focuses on:

- Retrieval relevance for different product attributes.
- Citation integrity and evidence traceability.
- Grounding of generated claims.
- Handling questions with potentially insufficient evidence.
- Response latency and token consumption.

Each generated answer and its supporting reviews will be inspected manually to identify factual inconsistencies and retrieval failures.


In [3]:
from time import perf_counter
import pandas as pd
test_questions = [
    "خریدارها درباره قدرت ماساژ این دستگاه چه نظری دارن؟",
    "کاربران درباره قابلیت مادون قرمز دستگاه چی گفتن؟",
    "آیا این دستگاه ضد آب است؟",
]

evaluation_results = []

for test_question in test_questions:
    start = perf_counter()

    result = qa_service.answer(
        test_question,
        product_id=21342,
        top_k=8,
    )

    latency = perf_counter() - start

    available_ids = {
        hit.item.review_id
        for hit in result.evidence
    }

    abstained = not result.cited_review_ids

    citation_valid = (
        None
        if abstained
        else set(result.cited_review_ids).issubset(available_ids)
    )

    assert citation_valid is not False

    evaluation_results.append(
        {
            "question": test_question,
            "retrieved_reviews": len(result.evidence),
            "cited_reviews": len(result.cited_review_ids),
            "citation_valid": citation_valid,
            "latency_seconds": round(latency, 2),
            "total_tokens": result.total_tokens,
            "abstained": abstained,
        }
    )

    print(f"\nQUESTION: {test_question}")
    print("-" * 70)
    print(result.answer)

    print("\nCITED EVIDENCE")

    for hit in result.evidence:
        if hit.item.review_id in result.cited_review_ids:
            print(f"\n[review_id:{hit.item.review_id}]")
            print(hit.item.body[:700])

    print("=" * 70)

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

print("\nMulti-Query RAG evaluation completed.")



QUESTION: خریدارها درباره قدرت ماساژ این دستگاه چه نظری دارن؟
----------------------------------------------------------------------
نظرات خریداران درباره قدرت ماساژ دستگاه "ماساژور برقی بیورر مدل MG70" متفاوت است:

1. برخی کاربران از قدرت ماساژ دستگاه راضی هستند و آن را با قدرت و کارآمد توصیف می‌کنند. به عنوان مثال، یکی از کاربران می‌گوید: "خیلی از خریدم راضیم چون هم راحت میشه ازش استفاده کرد هم اینکه خوب ماساژ میده. برعکس خیلی از ماساژور ها قدرت خوبی داره" [review_id:2183681] و کاربر دیگری اشاره می‌کند: "عالی ، با قدرت ، کارآمد ، خیلی راضیم ازش" [review_id:42335026].

2. در عوض، برخی دیگر از کاربران به ضعف قدرت ماساژ اشاره کرده‌اند. یکی از آن‌ها می‌گوید: "خیلی قدرت ضعیفی داره انگار اسباب بازیه" [review_id:41439382]. 

3. همچنین، یک کاربر دیگر به قدرت خوب دستگاه اشاره می‌کند اما در عین حال از کارایی مادون قرمزش نگران است و می‌گوید: "قدرت خوبی داره واقعا، اما کارایی مادون قرمزش رو نمیدونم دقیقا چی هست!" [review_id:2093806].

به طور کلی، نظرات متنوعی درباره قدرت ماساژ این دستگاه وجود د

,question,retrieved_reviews,cited_reviews,citation_valid,latency_seconds,total_tokens,abstained
0,خریدارها درباره قدرت ماساژ این دستگاه چه نظری ...,8,4,True,8.57,1425,False
1,کاربران درباره قابلیت مادون قرمز دستگاه چی گفتن؟,8,5,True,6.66,1250,False
2,آیا این دستگاه ضد آب است؟,8,0,None,2.52,1073,True



Multi-Query RAG evaluation completed.
